Using Numpy for Effficient Numerical Operations

Amani insurance case study

In [6]:
import time

import numpy as np
import pandas as pd

FILE_PATH = ("insurance_claims_messy.csv")

In [7]:
# Quick reused cleaning, same pattern as Part 3.
df = pd.read_csv(FILE_PATH)
df["claim_amount_kes"] = (
    df["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
)
df["claim_amount_kes"] = pd.to_numeric(df["claim_amount_kes"], errors="coerce").abs()
df["claim_type"] = df["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
group_median = df.groupby("claim_type")["claim_amount_kes"].transform("median")
df["claim_amount_kes"] = df["claim_amount_kes"].fillna(group_median)

# .to_numpy() converts the pandas column into a plain NumPy array of floats --
# this is the object all the vectorised operations below will run on.
amounts = df["claim_amount_kes"].to_numpy()
print(f"Loaded {len(amounts)} claim amounts as a NumPy array of dtype {amounts.dtype}")

Loaded 509 claim amounts as a NumPy array of dtype float64


In [8]:
# To make the timing gap obvious, repeat the array 2000 times over,
# simulating a much bigger claims book (~1 million values).
big_amounts = np.tile(amounts, 2000)
print(f"Simulating a bigger book: {len(big_amounts):,} claim amounts")


def loop_version(values):
    result = []
    for v in values:            # one Python-level iteration per value -- slow at scale
        result.append(v * 1.18)
    return result


start = time.perf_counter()
loop_result = loop_version(big_amounts)
loop_time = time.perf_counter() - start

start = time.perf_counter()
vectorised_result = big_amounts * 1.18   # the WHOLE array multiplied in one C-level operation
vector_time = time.perf_counter() - start

print(f"Plain Python loop:     {loop_time:.4f} seconds")
print(f"Vectorised NumPy:      {vector_time:.4f} seconds")
if vector_time > 0:
    print(f"Speedup: ~{loop_time / vector_time:.0f}x faster")

Simulating a bigger book: 1,018,000 claim amounts
Plain Python loop:     0.2286 seconds
Vectorised NumPy:      0.0154 seconds
Speedup: ~15x faster


In [9]:
# Sanity check: both approaches must compute IDENTICAL numbers -- the only
# difference is where the loop happens, not what gets calculated.
print("Spot check, first 3 values match:", np.allclose(loop_result[:3], vectorised_result[:3]))

Spot check, first 3 values match: True


In [10]:
# A "large claim" reserve-review threshold, e.g. anything over KES 500,000.
# amounts > 500_000 compares EVERY element at once, producing an array of
# True/False the same length as amounts.
large_claim_mask = amounts > 500_000

print(f"large_claim_mask dtype: {large_claim_mask.dtype}, shape: {large_claim_mask.shape}")
print(f"Number of large claims: {large_claim_mask.sum()}  (True counts as 1, so .sum() counts them)")
print(f"Total value of large claims: KES {amounts[large_claim_mask].sum():,.0f}")
print(f"Total value of ALL claims:   KES {amounts.sum():,.0f}")
print(f"Large claims are {amounts[large_claim_mask].sum() / amounts.sum() * 100:.1f}% "
      f"of total claims value, from just {large_claim_mask.mean() * 100:.1f}% of claims.")

large_claim_mask dtype: bool, shape: (509,)
Number of large claims: 155  (True counts as 1, so .sum() counts them)
Total value of large claims: KES 150,608,900
Total value of ALL claims:   KES 202,487,550
Large claims are 74.4% of total claims value, from just 30.5% of claims.


In [11]:
# Categorise every claim into a risk band in one line, no loop at all.
risk_band = np.where(
    amounts > 1_000_000, "High",
    np.where(amounts > 200_000, "Medium", "Low")
)

# np.unique with return_counts=True gives us each distinct label plus how
# many times it appears -- a quick way to tally categories from an array.
bands, counts = np.unique(risk_band, return_counts=True)
print("Risk band counts:")
for b, c in zip(bands, counts):
    print(f"  {b:8s} {c}")

Risk band counts:
  High     41
  Low      249
  Medium   219


In [12]:
# Parse claim_date into an actual date, then pull out the month number.
df["claim_date_parsed"] = pd.to_datetime(df["claim_date"], errors="coerce", format="mixed")
df["month"] = df["claim_date_parsed"].dt.month
df["region_clean"] = df["region"].astype(str).str.strip().str.title()

# pivot_table builds a region x month table of summed claim values.
# .to_numpy() exposes the genuinely 2D NumPy array sitting underneath it.
pivot = df.pivot_table(
    values="claim_amount_kes", index="region_clean", columns="month",
    aggfunc="sum", fill_value=0,
)
matrix = pivot.to_numpy()
print(f"Matrix shape: {matrix.shape}  (regions x months)")

Matrix shape: (6, 12)  (regions x months)


In [13]:
# axis=0 collapses ROWS   (moves down each column) -> one total per COLUMN (month)
# axis=1 collapses COLUMNS (moves across each row)   -> one total per ROW (region)
monthly_totals = matrix.sum(axis=0)
region_totals = matrix.sum(axis=1)

print("Total claims value per month, all regions (matrix.sum(axis=0)):")
for month_num, total in zip(pivot.columns, monthly_totals):
    print(f"  Month {month_num:>2}: KES {total:>14,.0f}")

Total claims value per month, all regions (matrix.sum(axis=0)):
  Month  1: KES     25,506,300
  Month  2: KES     20,285,300
  Month  3: KES     21,115,500
  Month  4: KES     29,988,250
  Month  5: KES     14,796,350
  Month  6: KES     17,675,150
  Month  7: KES     26,599,800
  Month  8: KES     19,495,350
  Month  9: KES     15,350,250
  Month 10: KES      2,899,100
  Month 11: KES      6,311,100
  Month 12: KES      2,465,100


In [14]:
print("Total claims value per region, all months (matrix.sum(axis=1)):")
for region, total in zip(pivot.index, region_totals):
    print(f"  {region:10s} KES {total:>14,.0f}")

Total claims value per region, all months (matrix.sum(axis=1)):
  Eldoret    KES     32,118,500
  Kisumu     KES     42,083,600
  Mombasa    KES     30,060,650
  Nairobi    KES     23,997,700
  Nakuru     KES     31,711,100
  Nyeri      KES     42,516,000
